other sol: https://github.com/stefanasandei/roai-solved/blob/main/contests/pre-iaio-2026/hidden-noise.ipynb
problem: https://judge.nitro-ai.org/competitions/nitro/pre-iaio-2026/1/view

In [2]:
import json
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# 1. Load the raw numbers
with open("model_params.json", "r") as f:
    params = json.load(f)

# 2. Create a brand new, empty pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# 3. Inject the "params" back into the components
pipe.named_steps['scaler'].mean_ = np.array(params["scaler_mean"])
pipe.named_steps['scaler'].var_ = np.array(params["scaler_var"])
pipe.named_steps['scaler'].scale_ = np.array(params["scaler_scale"])
pipe.named_steps['scaler'].n_features_in_ = len(params["scaler_mean"])

pipe.named_steps['regressor'].coef_ = np.array(params["coef"])
pipe.named_steps['regressor'].intercept_ = params["intercept"]

print("Pipeline reconstructed successfully!")

pipe

Pipeline reconstructed successfully!


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None


In [4]:
import pandas as pd

train_df = pd.read_csv('train_data.csv').drop(['datapointID', 'subtaskID'], axis=1)

train_df.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,target
0,38.386698,96.119755,74.799984,60.720818,17.224338,16.089440,7.296285,88.208061,60.737244,71.793142,-3.316884
1,7.509907,103.118333,92.599548,26.231124,27.665694,21.204378,39.120991,61.771641,46.851906,34.885306,-14.268649
2,57.722605,10.057386,23.272128,33.462029,39.583413,76.698472,14.443321,45.518765,56.918328,0.984856,-20.099994
3,62.298347,18.787687,9.154591,96.303772,99.248860,81.650804,32.924315,12.399853,69.459085,45.647168,-11.175629
4,14.072725,51.618305,6.646087,92.645215,29.129085,67.234057,34.152586,55.193712,55.924881,20.460944,-21.376248


In [49]:
x = train_df.drop('target', axis=1).to_numpy()
y = train_df['target'].to_numpy()

In [ ]:
n = len(params["scaler_mean"])
s = np.zeros((n,n))
np.fill_diagonal(s, 1.0/np.array(params["scaler_scale"]))
w = np.array(params['coef'])

w_norm = w.T @ s
w_norm = w_norm.reshape(1, -1)
b_norm = np.array(params['intercept'] - w_norm @ np.array(params["scaler_mean"]))

lamda = (y - (w_norm @ x.T + b_norm)) / np.dot(w_norm, w_norm.T)
real_x = x + lamda.T @ w_norm

errs = pipe.predict(real_x)-y

array([-1.19904087e-14, -7.10542736e-15, -1.42108547e-14, -1.24344979e-14,
        0.00000000e+00,  2.66453526e-15, -8.88178420e-15, -7.10542736e-15,
       -4.44089210e-15, -4.44089210e-15, -8.88178420e-15, -1.19904087e-14,
       -7.10542736e-15, -1.73194792e-14, -1.06581410e-14, -4.44089210e-15,
       -1.77635684e-14, -1.06581410e-14, -5.10702591e-15,  3.55271368e-15,
       -7.10542736e-15, -7.99360578e-15, -1.15463195e-14, -1.06581410e-14,
       -1.77635684e-15, -1.77635684e-15, -2.66453526e-15, -1.06581410e-14,
       -7.10542736e-15, -1.06581410e-14, -1.33226763e-14, -7.10542736e-15,
       -1.50990331e-14, -3.55271368e-15, -1.06581410e-14, -1.06581410e-14,
       -4.44089210e-15, -1.42108547e-14, -1.77635684e-15, -8.88178420e-16,
       -7.10542736e-15, -1.68753900e-14, -8.88178420e-15,  0.00000000e+00,
       -1.33226763e-14, -7.10542736e-15, -8.88178420e-15, -1.06581410e-14,
       -1.50990331e-14, -7.10542736e-15, -1.24344979e-14, -8.88178420e-15,
       -8.88178420e-16, -

In [105]:
pipe.score(real_x, train_df["target"])

0.9276330690013188